In [0]:
%run ../utils/adls_auth

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from datetime import datetime


schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
])

# Create the bad rows
bad_rows = [
    # 1. Null pickup datetime
    Row(
        VendorID=1,
        tpep_pickup_datetime=None,
        tpep_dropoff_datetime=datetime(2025, 6, 1, 10, 0),
        passenger_count=1,
        trip_distance=2.0,
        RatecodeID=1,
        store_and_fwd_flag="N",
        PULocationID=100,
        DOLocationID=200,
        payment_type=1,
        fare_amount=10.0,
        extra=0.0,
        mta_tax=0.5,
        tip_amount=1.0,
        tolls_amount=0.0,
        improvement_surcharge=0.3,
        total_amount=11.8,
        congestion_surcharge=0.0,
        Airport_fee=None,
        cbd_congestion_fee=None,
        year=2025,
        month=6
    ),
    # 2. Impossible passenger_count
    Row(
        VendorID=2,
        tpep_pickup_datetime=datetime(2025, 6, 1, 9, 0),
        tpep_dropoff_datetime=datetime(2025, 6, 1, 9, 30),
        passenger_count=50,
        trip_distance=3.0,
        RatecodeID=1,
        store_and_fwd_flag="N",
        PULocationID=100,
        DOLocationID=200,
        payment_type=1,
        fare_amount=15.0,
        extra=0.0,
        mta_tax=0.5,
        tip_amount=1.0,
        tolls_amount=0.0,
        improvement_surcharge=0.3,
        total_amount=16.8,
        congestion_surcharge=0.0,
        Airport_fee=None,
        cbd_congestion_fee=None,
        year=2025,
        month=6
    ),
    # 3. Negative trip_distance
    Row(
        VendorID=1,
        tpep_pickup_datetime=datetime(2025, 6, 1, 11, 0),
        tpep_dropoff_datetime=datetime(2025, 6, 1, 11, 15),
        passenger_count=2,
        trip_distance=-999.0,
        RatecodeID=1,
        store_and_fwd_flag="N",
        PULocationID=100,
        DOLocationID=200,
        payment_type=1,
        fare_amount=5.0,
        extra=0.0,
        mta_tax=0.5,
        tip_amount=0.0,
        tolls_amount=0.0,
        improvement_surcharge=0.3,
        total_amount=5.8,
        congestion_surcharge=0.0,
        Airport_fee=None,
        cbd_congestion_fee=None,
        year=2025,
        month=6
    ),
]

# Create DataFrame with explicit schema
bad_df = spark.createDataFrame(bad_rows, schema=schema)

# Write to Bronze (append to June 2025)
bad_df.write.mode("append").parquet(
    "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/trips_raw/year=2025/month=06"
)

print(f"Injected {len(bad_rows)} intentionally bad rows into June 2025 Bronze trips.")
bad_df.display()

In [0]:
from pyspark.sql.functions import col
# Check quarantine for the injected rows
quarantine_df = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_quarantine")

# Count injected bad rows in quarantine
injected_found = quarantine_df.filter(
    (col("passenger_count") == 50) |
    (col("trip_distance") == -999.0) |
    (col("tpep_pickup_datetime").isNull())
).count()

print(f"Injected bad rows found in quarantine: {injected_found}")


# Show them
quarantine_df.filter(
    (col("passenger_count") == 50) |
    (col("trip_distance") == -999.0) |
    (col("tpep_pickup_datetime").isNull())
).display()